# IEEE-CIS Fraud Detection - Model Inference

## Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub

os.environ['MLFLOW_TRACKING_USERNAME'] = 'sansi23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'YOUR_DAGSHUB_TOKEN'

dagshub.init(
    repo_owner='sansi23',
    repo_name='IEEE-CIS-Fraud-Detection',
    mlflow=True
)

print('MLflow URI:', mlflow.get_tracking_uri())

## Load Best Model

In [ ]:
model_name    = 'XGBoost_FraudDetection_Pipeline'
model_version = 1

model_uri = f'models:/{model_name}/{model_version}'
pipeline  = mlflow.sklearn.load_model(model_uri)

print('Model loaded  :', type(pipeline))
print('Pipeline steps:', [name for name, _ in pipeline.steps])

## Load Test Data

No preprocessing needed — the pipeline handles everything internally.

In [ ]:
DATA_DIR = 'data/'  # adjust path if needed

test_transaction = pd.read_csv(DATA_DIR + 'test_transaction.csv')
test_identity    = pd.read_csv(DATA_DIR + 'test_identity.csv')

test     = test_transaction.merge(test_identity, on='TransactionID', how='left')
test_ids = test['TransactionID']
X_test   = test.drop(columns=['TransactionID'])

print('X_test shape:', X_test.shape)

## Generate Predictions & Submission

In [ ]:
y_proba = pipeline.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud'      : y_proba
})

submission.to_csv('submission.csv', index=False)

print(submission.shape)
print(submission.head())